## WARNING
Important:
**BERTopic generates topic IDs dynamically at each run.**
Therefore, topic numbers and associated results may differ across executions, even with the same input data.
This notebook is meant for reading and understanding the code
rather than being re-run to reproduce identical results.


In [ ]:
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
ROOT = HERE if (HERE / "data").exists() else HERE.parent

csv_path = ROOT / "data" / "nlp_transformed" / "01_sentiment_analysis_done.csv"
df = pd.read_csv(csv_path, encoding="utf-8")
df.sample(10)

---
# PART 5 : Topic detection with BERTopic
---

**Explanations**

The objective of this step is to:

1.   Identify the topics present in each comment
2.   Group these topics into a set of main themes (approximately fifteen, such as compensation, management, work environment, etc.)
2.   Create two columns, topics_positive and topics_negative, listing the associated topics for each comment

## 5.1 Install the libraries

In [3]:
!pip -q install bertopic sentence-transformers umap-learn hdbscan

import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.6 MB/s eta 0:00:00


## 5.2 Keep only comments

In [9]:
# BERTopic operates exclusively on textual data.
df_text = df[df["has_comment"] == True].copy()
df_text = df_text[df_text["comment"].notna()].copy()  # ensure that a comment is actually present
# (using notna as a safety check, in case has_comment is incorrectly specified)

# copy() creates a new, independent DataFrame object in memory
df_text["comment"] = (
    df_text["comment"]
    .astype(str)        # ensure the content is treated as text
    .str.strip()        # remove leading and trailing whitespaces
)

# Convert the comment column into a Python list (required input format for BERTopic)
docs = df_text["comment"].tolist()

print("Number of comments used:", len(docs))

Number of comments used: 3747


## 5.3 Create and train BERTopic

In [10]:
# ========================= WARNING ========================
# Important: if you run this code again, topic IDs may change/order differently each time.
# This is also the longest step to execute.
# ========================= WARNING ========================


# BERTopic groups similar commentcomments.
# min_topic_size prevents the creation of too many micro-topics.
from bertopic import BERTopic

topic_model = BERTopic(
    language="english",
    min_topic_size=30
    # min_topic_size = minimum number of documents (comment comments)
    # required for a group to be considered a valid topic
)

# The model learns topics from the comment comments
topics, probs = topic_model.fit_transform(docs)

# topics = list/array where each value corresponds to a topic_id
# 0, 3, 5 = valid topics (BERTopic clusters comments and automatically assigns numeric IDs)
# -1 = outlier (does not belong to any clearly defined topic)
# probs = probability associated with the assignment, indicating the model's confidence

# Each comment comment is assigned a topic_id (numeric identifier)
df_text["topic_id"] = topics

print("Number of unique topics:", len(set(topics)))
print("Number of outliers (topic = -1):", (df_text["topic_id"] == -1).sum())

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of unique topics: 66
Number of outliers (topic = -1): 278


## 5.4 Assign human-readable labels to each topic (keywords)

In [11]:
# A topic_id alone is not informative; a label is created using the main keywords.
def topic_to_label(topic_id, n_words=4):
    if topic_id == -1:
        return ""  # outlier -> no label
    keywords = [w for w, _ in topic_model.get_topic(topic_id)[:n_words]]
    return "; ".join(keywords)

# The four most representative keywords are retained to construct the topic label
df_text["topic_label"] = df_text["topic_id"].apply(
    lambda t: topic_to_label(t, n_words=4)
)

# Topic overview
topic_info = topic_model.get_topic_info()
topic_info.head(100)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,278,-1_would_logistics_support_setup,"[would, logistics, support, setup, well, smoot...","[Tools and logistics support delivery well, an..."
1,0,110,0_attractive_reputation_remains_solid,"[attractive, reputation, remains, solid, inter...",[Skapa remains attractive with interesting mis...
2,1,101,1_competitive_hesitation_recommend_without,"[competitive, hesitation, recommend, without, ...",[Compensation feels competitive for the market...
3,2,100,2_integration_onboarding_were_welcoming,"[integration, onboarding, were, welcoming, smo...","[Onboarding and integration were smooth, and t..."
4,3,96,3_like_regular_from_expectations,"[like, regular, from, expectations, coaching, ...",[I would like more regular coaching and cleare...
...,...,...,...,...,...
61,60,35,60_build_learning_skills_help,"[build, learning, skills, help, opportunities,...",[Learning opportunities and projects help me b...
62,61,35,61_commitment_respectful_responsible_genuine,"[commitment, respectful, responsible, genuine,...",[There is a genuine commitment to responsible ...
63,62,35,62_progress_chances_merit_based,"[progress, chances, merit, based, see, impact,...",[I see real chances to progress based on merit...
64,63,31,63_increased_decisions_transparency_direction,"[increased, decisions, transparency, direction...",[I appreciate the increased transparency from ...


## 5.5 Derive macro-themes from BERTopic outputs by mapping topic_id to macro_theme

In [13]:
# Objective:
# - BERTopic has already been trained and a topic_id has been assigned to each comment.
# - These topics are now grouped into approximately 15 stable "HR macro-themes"-

topic_to_macro = {
    -1: "",  # outlier

    # Company attractiveness
    0:  "Company attractiveness",  # attractive missions, solid reputation
    9:  "Company attractiveness",  # employer brand strength
    36: "Company attractiveness",  # company stable, well-positioned, strong demand
    40: "Company attractiveness",  # attractive missions, reputation
    46: "Company attractiveness",  # stability, market positioning
    57: "Company attractiveness",  # attractiveness, missions, reputation

    # Compensation
    1:  "Compensation",  # compensation competitive
    4:  "Compensation",  # compensation competitive
    5:  "Compensation",  # pay progression slow vs responsibilities/performance
    15: "Compensation",  # compensation not keeping pace with inflation/benchmarks
    24: "Compensation",  # compensation competitive
    26: "Compensation",  # satisfaction with pay and benefits
    27: "Compensation",  # satisfaction with pay and benefits
    28: "Compensation",  # compensation not keeping pace with inflation/benchmarks
    33: "Compensation",  # satisfaction with pay and benefits
    47: "Compensation",  # slow pay progression

    # Integration
    2:  "Integration",  # onboarding and integration smooth
    12: "Integration",  # onboarding and integration smooth
    23: "Integration",  # onboarding and integration smooth

    # Management practices
    3:  "Management practices",  # coaching, expectations, inclusion/equity
    6:  "Management practices",  # top-down decision-making, lack of recognition
    18: "Management practices",  # inconsistent management, late feedback
    20: "Management practices",  # consistent recognition and coaching
    22: "Management practices",  # supportive leadership, clear feedback
    30: "Management practices",  # autonomy, trust, empowering style
    41: "Management practices",  # supportive leadership, clear feedback
    43: "Management practices",  # autonomy and trust
    45: "Management practices",  # autonomy and trust
    52: "Management practices",  # consistent recognition and coaching
    63: "Management practices",  # leadership transparency on decisions

    # Tools
    13: "Tools",  # digital tools and processes improving
    48: "Tools",  # tools and logistics support delivery
    53: "Tools",  # tools and logistics support delivery

    # Training
    8:  "Training",  # learning opportunities, skill development
    14: "Training",  # learning opportunities, skill development
    31: "Training",  # training available but time constrained
    60: "Training",  # learning opportunities, skill development

    # Career progression
    25: "Career progression",  # career paths clearer, internal mobility
    34: "Career progression",  # unclear promotion criteria
    35: "Career progression",  # career paths clearer
    37: "Career progression",  # structured development, growth transparency
    54: "Career progression",  # career paths clearer
    62: "Career progression",  # progression based on merit and impact
    29: "Career progression",  # progression based on merit and impact

    # Innovation & competitiveness
    19: "Innovation & competitiveness",  # concern about adapting to market changes
    44: "Innovation & competitiveness",  # innovation and forward-looking strategy
    49: "Innovation & competitiveness",  # innovation and strategy

    # Purpose & meaning
    10: "Purpose & meaning",  # mission variety, client exposure
    32: "Purpose & meaning",  # loss of meaning due to utilization focus
    38: "Purpose & meaning",  # meaningful impact for clients and society
    61: "Purpose & meaning",  # responsible consulting, respectful culture

    # Work conditions
    58: "Work conditions",  # work conditions good, hybrid handled pragmatically
    59: "Work conditions",  # work conditions good, hybrid handled pragmatically

    # Workload
    17: "Workload",  # workload peaks, staffing tight
    56: "Workload",  # overtime becoming default

    # Quality of Work Life
    16: "Quality of Work Life",  # fatigue and burnout risk
    21: "Quality of Work Life",  # work-life balance, flexibility
    42: "Quality of Work Life",  # work-life balance, flexibility
    64: "Quality of Work Life",  # balance and flexibility

    # Team climate
    7:  "Team climate",  # friendly and supportive atmosphere
    11: "Team climate",  # friendly and supportive atmosphere
    39: "Team climate",  # team cohesion varies, isolation between assignments
    50: "Team climate",  # team spirit and atmosphere
    51: "Team climate",  # team spirit

    # Internal organization
    55: "Internal organization",  # bench periods and project allocation consistency
}


df_text["macro_theme_topic"] = df_text["topic_id"].map(topic_to_macro).fillna("")


## 5.6 Detect multiple macro-themes within a single comment (sentence-level analysis)

Many comments contain multiple topics; analyzing them as a whole may assign only one.
Comments are therefore split into sentences, and a macro-theme is predicted for each sentence.


In [28]:
# ============================================================
# Cleaning and sentence-by-sentence splitting
# ============================================================

import re
import nltk
from nltk.tokenize import sent_tokenize

# Required downloads for Colab
nltk.download("punkt")
nltk.download("punkt_tab")

# Light cleaning to avoid empty sentences and reduce multiple spaces.
def clean_sentence(s):
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s


# 1) Build a sentence-level DataFrame

df_sent = df_text[["survey_response_id", "comment", "sentiment"]].copy()

df_sent["sentence"] = df_sent["comment"].apply(
    lambda x: [
        clean_sentence(s)
        for s in sent_tokenize(str(x))
        if clean_sentence(s)
    ]
)

df_sent = df_sent.explode("sentence").dropna(subset=["sentence"]).copy()
df_sent = df_sent[df_sent["sentence"].str.len() > 0].copy()


# 2) Predict a BERTopic topic for each sentence

sentence_topics, _ = topic_model.transform(df_sent["sentence"].tolist())
df_sent["sentence_topic"] = sentence_topics


# 3) Convert the topic into a macro-theme

df_sent["sentence_macro"] = (
    df_sent["sentence_topic"]
        .map(topic_to_macro)
        .fillna("")
)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [30]:
# pour voir les phrases qui n'ont pas de macro-thèmes affectés
df_unmapped = df_sent[df_sent["sentence_macro"] == ""][["sentence"]].drop_duplicates()
df_unmapped

,sentence
26,"I feel aligned with the values, especially inc..."
40,Cross-team collaboration is improving and it’s...
62,Recent uncertainty in priorities makes the lon...
197,It would help to see concrete actions and foll...


In [31]:
############# 3 bis Post-review customization ############

# 0) Create a cleaned version of the sentence (avoid repeating strip/astype everywhere)
df_sent["sentence_clean"] = df_sent["sentence"].astype(str).str.strip()


# 1) Exclude generic sentences (set macro-theme to empty)
GENERIC_SENTENCES = [
    "Overall, I would recommend Skapa without hesitation.",
    "Addressing this would make a big difference for engagement.",
]

df_sent.loc[
    df_sent["sentence_clean"].isin(GENERIC_SENTENCES),
    "sentence_macro"
] = ""


# 2) Exact sentence -> macro-theme overrides
SENTENCE_TO_MACRO = {
    # New macro-theme
    "I feel proud to be part of the company.": "Pride of belonging",

    # Governance & decision-making
    "Recent uncertainty in priorities makes the long-term direction feel less clear.":
        "Governance & decision-making",
    "Strategic priorities shift often and it can be hard to understand the direction.":
        "Governance & decision-making",
    "I would like to see stronger action on inclusion and equity, not just messaging.":
        "Governance & decision-making",
    "I would like more visibility from leadership and less pressure driven by short-term targets.":
        "Governance & decision-making",
    "Recent uncertainty in priorities makes the long-term direction feel less clear.":
        "Governance & decision-making",

    # Tools
    "Digital tools and processes are improving and make delivery more efficient.": "Tools",
    "Some tools and processes are slow and create friction in day-to-day delivery.": "Tools",

    # Internal organization
    "Internal collaboration can feel siloed and handoffs between teams are not always smooth.": "Internal organization",

    # Management practices
    "Supportive leadership and clear feedback make it easy to do great work.": "Management practices",
    "From a Valencia perspective, Supportive leadership and clear feedback make it easy to do great work.": "Management practices",

    # Quality of Work Life
    "Work-life balance could improve; overtime can become the default during delivery.": "Quality of Work Life",

    # Purpose & meaning
    "I feel aligned with the values, especially inclusion, sustainability and fairness.": "Purpose & meaning",

    # Internal organization
    "Cross-team collaboration is improving and it’s easier to work transversally.": "Internal organization",
}

for sentence, macro in SENTENCE_TO_MACRO.items():
    df_sent.loc[
        df_sent["sentence_clean"] == sentence,
        "sentence_macro"
    ] = macro

In [32]:
# pour voir les phrases qui n'ont pas de macro-thèmes affectés
df_unmapped = df_sent[df_sent["sentence_macro"] == ""][["sentence"]].drop_duplicates()
df_unmapped

,sentence
11,"Overall, I would recommend Skapa without hesit..."
197,It would help to see concrete actions and foll...
667,Addressing this would make a big difference fo...


In [33]:
# 4) Remove unmapped sentences
df_sent = df_sent[df_sent["sentence_macro"] != ""].copy()

In [34]:
# Checking
df_sent[["sentence", "sentence_macro"]].drop_duplicates(subset="sentence")


,sentence,sentence_macro
0,Team spirit is strong and the atmosphere is fr...,Team climate
0,I feel proud to be part of the company.,Pride of belonging
1,Pay progression feels slow compared with respo...,Compensation
1,"Career progression can feel unclear, especiall...",Career progression
2,Innovation and a forward-looking strategy make...,Innovation & competitiveness
...,...,...
4054,"Based in Stockholm, I can maintain a healthy b...",Quality of Work Life
4122,"Here in our Malmö office, I see real chances t...",Career progression
4221,"In our Gothenburg offices, I worry about compe...",Innovation & competitiveness
4277,"From a Seville perspective, Tools and logistic...",Tools


In [59]:
df_sent["sentence_macro"].value_counts(normalize=True)*100

,proportion
sentence_macro,
Management practices,17.508251
Compensation,12.326733
Pride of belonging,11.369637
Company attractiveness,7.772277
Career progression,7.755776
Purpose & meaning,7.607261
Quality of Work Life,6.435644
Team climate,5.016502
Innovation & competitiveness,4.471947


## 5.7 Create topics_positive and topics_negative (handling "to be determined" comments)

In [40]:
# Objective:
# - Create two columns:
#   - topics_positive: list of positive themes (separated by "; ")
#   - topics_negative: list of negative themes (separated by "; ")
#
# Rules:
# - If sentiment = "positive" -> all themes from the comment go into topics_positive
# - If sentiment = "negative" -> all themes from the comment go into topics_negative
# - If sentiment = "to be determined":
#     -> split sentence by sentence:
#         * sentences with a negative signal -> themes go into topics_negative
#         * other sentences -> themes go into topics_positive

negative_cues = [

    # --- Requests / polite frustration ---
    "would like",
    "like more",
    "could be",
    "not always",
    "it would help",
    "it would be helpful",

    # --- Compensation / financial rewards ---
    "feels slow",
    "slow compared",
    "pace with inflation",
    "inflation",
    "market benchmarks",
    "pay progression",
    "compensation has not kept",
    "not kept pace",

    # --- Workload / burnout ---
    "sustained pressure",
    "pressure",
    "pressure can lead",
    "lead to fatigue",
    "fatigue",
    "burnout",
    "early signs of burnout",
    "overtime",
    "overtime can become",
    "can be intense",
    "Workload peaks",
    "peaks",

    # --- Management / leadership ---
    "top down",
    "inconsistent",
    "too late",
    "late",
    "clearer expectations",
    "more transparency",
    "feedback is often too late",
    "management practices feel inconsistent",

    # --- Career development / progression ---
    "unclear",
    "criteria for promotion",
    "promotion criteria",
    "career progression can feel unclear",
    "limited growth",
    "limited opportunities",

    # --- Organization / processes ---
    "processes are slow",
    "slow",
    "create friction",
    "siloed",
    "bench periods",
    "project allocation",

    # --- Meaning / purpose of work ---
    "less meaningful",
    "not meaningful",
    "meaningful when",
    "driven mainly by utilization",

    # --- Work-life balance ---
    "work life balance could improve",
    "balance could improve",

    # --- Uncertainty / context ---
    "uncertainty",
    "limited",
    "worry about",

    # --- Contrastive connectors ---
    "but",
    "however",
    "although",
    "though"
]

negative_cues

def sentence_has_negative_signal(text):
    """Détecte si une phrase contient au moins un signal négatif."""
    text = str(text).lower()
    return any(kw in text for kw in negative_cues)

df_sent["sentence_is_negative"] = df_sent["sentence"].apply(sentence_has_negative_signal)

def join_unique(values):
    """Transforme une liste de thèmes en texte 'A; B; C' sans doublons."""
    vals = sorted(set([v for v in values if v]))
    return "; ".join(vals)

# df_topics: one row per comment (survey_response_id) with topics_positive / topics_negative
df_topics = df_text[["survey_response_id", "sentiment"]].copy()
df_topics["topics_positive"] = ""
df_topics["topics_negative"] = ""


# --- 5.7.1 Case "positive": all topics go into topics_positive
pos_ids = df_topics.loc[df_topics["sentiment"] == "positive", "survey_response_id"]
pos_map = (df_sent[df_sent["survey_response_id"].isin(pos_ids)]
           .groupby("survey_response_id")["sentence_macro"]
           .apply(join_unique))

df_topics.loc[df_topics["sentiment"] == "positive", "topics_positive"] = (
    df_topics.loc[df_topics["sentiment"] == "positive", "survey_response_id"].map(pos_map).fillna("")
)

# --- 5.7.2 Case "negative": all topics go into topics_negative
neg_ids = df_topics.loc[df_topics["sentiment"] == "negative", "survey_response_id"]
neg_map = (df_sent[df_sent["survey_response_id"].isin(neg_ids)]
           .groupby("survey_response_id")["sentence_macro"]
           .apply(join_unique))

df_topics.loc[df_topics["sentiment"] == "negative", "topics_negative"] = (
    df_topics.loc[df_topics["sentiment"] == "negative", "survey_response_id"].map(neg_map).fillna("")
)

# --- 5.7.3 "To be determined" case: split sentence by sentence
mix_ids = df_topics.loc[df_topics["sentiment"] == "to be determined", "survey_response_id"]
mix_df = df_sent[df_sent["survey_response_id"].isin(mix_ids)].copy()

mix_pos_map = (mix_df[mix_df["sentence_is_negative"] == False]
               .groupby("survey_response_id")["sentence_macro"]
               .apply(join_unique))

mix_neg_map = (mix_df[mix_df["sentence_is_negative"] == True]
               .groupby("survey_response_id")["sentence_macro"]
               .apply(join_unique))

df_topics.loc[df_topics["sentiment"] == "to be determined", "topics_positive"] = (
    df_topics.loc[df_topics["sentiment"] == "to be determined", "survey_response_id"].map(mix_pos_map).fillna("")
)

df_topics.loc[df_topics["sentiment"] == "to be determined", "topics_negative"] = (
    df_topics.loc[df_topics["sentiment"] == "to be determined", "survey_response_id"].map(mix_neg_map).fillna("")
)

# Quick check on "to be determined" cases
df_topics[df_topics["sentiment"] == "to be determined"][["topics_positive", "topics_negative"]].head(20)



,topics_positive,topics_negative
39,,Company attractiveness; Purpose & meaning
46,Career progression,
81,Integration,Management practices
109,Team climate,Internal organization
114,Integration,Management practices
117,Career progression,
141,Career progression,
147,Team climate,Compensation
173,Career progression,
192,,Company attractiveness


In [61]:
# Remove existing topics_positive/topics_negative columns before step 5.8 (if already present)

df = df.drop(columns=["topics_positive", "topics_negative", "sentiment"], errors="ignore")

## 5.8 Reintegrate topics_positive and topics_negative into the full DataFrame

In [62]:
df = df.merge(
    df_topics[["survey_response_id", "topics_positive", "topics_negative", "sentiment"]],
    on="survey_response_id",
    how="left"
)

# For responses without a comment, empty strings are assigned
df["topics_positive"] = df["topics_positive"].fillna("")
df["topics_negative"] = df["topics_negative"].fillna("")

# Examples
df[["sentiment", "topics_positive", "topics_negative", "comment"]].sample(10, random_state=42)


,sentiment,topics_positive,topics_negative,comment
4058,positive,Internal organization; Pride of belonging; Pur...,,Cross-team collaboration is improving and it’s...
315,negative,,Workload,Workload peaks can be intense and staffing fee...
3093,positive,Innovation & competitiveness; Team climate,,Team spirit is strong and the atmosphere is fr...
3198,positive,Compensation; Management practices,,Supportive leadership and clear feedback make ...
2813,to be determined,Career progression,,Career paths are becoming clearer and internal...
755,positive,Management practices; Quality of Work Life,,Supportive leadership and clear feedback make ...
387,positive,Team climate,,Team spirit is strong and the atmosphere is fr...
1780,negative,,Workload,Workload peaks can be intense and staffing fee...
594,negative,,Compensation,Compensation has not kept pace with inflation ...
1588,negative,,Management practices,Management practices feel inconsistent and fee...


## 5.8 Handling "to be determined" cases

In [63]:
# Creation of "empty / non-empty" indicators

mask_undetermined = df["sentiment"] == "to be determined"

has_pos = df["topics_positive"].fillna("").str.strip() != ""
has_neg = df["topics_negative"].fillna("").str.strip() != ""

In [64]:
df.loc[
    mask_undetermined & has_pos & ~has_neg,  # if topics_positive is filled and topics_negative is empty, then sentiment is positive
    "sentiment"
] = "positive"

df.loc[
    mask_undetermined & ~has_pos & has_neg,  # if topics_negative is filled and topics_positive is empty, then sentiment is negative
    "sentiment"
] = "negative"

df.loc[
    mask_undetermined & has_pos & has_neg,   # if both columns are filled, then sentiment is mixed
    "sentiment"
] = "mixed"

In [65]:
(df["sentiment"].value_counts(normalize=True) * 100).round(2)

,proportion
sentiment,
positive,68.45
negative,30.29
mixed,1.25


In [66]:
df[["comment", "sentiment", "topics_positive", "topics_negative"]][df["sentiment"] == "mixed"].drop_duplicates(subset=["comment"])

,comment,sentiment,topics_positive,topics_negative
81,I would like more regular coaching and clearer...,mixed,Integration,Management practices
109,Team spirit is strong and the atmosphere is fr...,mixed,Team climate,Internal organization
114,"Onboarding and integration were smooth, and th...",mixed,Integration,Management practices
147,Team spirit is strong and the atmosphere is fr...,mixed,Team climate,Compensation
293,Team spirit is strong and the atmosphere is fr...,mixed,Team climate,Purpose & meaning
295,"Onboarding and integration were smooth, and th...",mixed,Integration,Quality of Work Life
492,"Onboarding and integration were smooth, and th...",mixed,Integration,Purpose & meaning
501,Team spirit is strong and the atmosphere is fr...,mixed,Team climate,Quality of Work Life
686,Team spirit is strong and the atmosphere is fr...,mixed,Team climate,Quality of Work Life
771,Pay progression feels slow compared with respo...,mixed,Team climate,Compensation


In [47]:
df[["sentiment", "topics_positive", "topics_negative", "comment"]]

,sentiment,topics_positive,topics_negative,comment
0,positive,Pride of belonging; Team climate,,Team spirit is strong and the atmosphere is fr...
1,negative,,Career progression; Compensation,Pay progression feels slow compared with respo...
2,positive,Innovation & competitiveness; Management pract...,,Innovation and a forward-looking strategy make...
3,positive,Management practices,,Supportive leadership and clear feedback make ...
4,positive,Innovation & competitiveness; Management pract...,,Innovation and a forward-looking strategy make...
...,...,...,...,...
4386,positive,Quality of Work Life; Team climate,,I can maintain a healthy balance most of the t...
4387,positive,Management practices; Team climate,,I appreciate the increased transparency from l...
4388,negative,,Management practices,I would like more regular coaching and clearer...
4389,positive,Compensation,,Compensation feels competitive for the market ...


## 5.9 Quick checks (top positive / negative themes)

In [67]:
# Distribution of positive comments

pos_pct = (
    df.loc[df["topics_positive"].notna() & (df["topics_positive"].str.strip() != ""), "topics_positive"]
      .str.split(r"\s*;\s*")     # split sur ;
      .explode()
      .str.strip()
      .value_counts(normalize=True)
      .mul(100)
      .round(1)
      .reset_index()
)

pos_pct.columns = ["macro_theme", "percentage"]
pos_pct

,macro_theme,percentage
0,Management practices,16.3
1,Pride of belonging,15.8
2,Compensation,11.1
3,Purpose & meaning,8.8
4,Company attractiveness,8.5
5,Career progression,7.4
6,Team climate,5.7
7,Integration,5.2
8,Quality of Work Life,5.0
9,Tools,5.0


In [69]:
# Distribution of negative comments

neg_pct = (
    df.loc[df["topics_negative"].notna() & (df["topics_negative"].str.strip() != ""), "topics_negative"]
      .str.split(r"\s*;\s*")
      .explode()
      .str.strip()
      .value_counts(normalize=True)
      .mul(100)
      .round(1)
      .reset_index()
)

neg_pct.columns = ["macro_theme", "percentage"]
neg_pct

,macro_theme,percentage
0,Management practices,20.4
1,Compensation,15.7
2,Quality of Work Life,10.2
3,Governance & decision-making,8.9
4,Career progression,8.7
5,Innovation & competitiveness,6.0
6,Internal organization,5.9
7,Company attractiveness,5.9
8,Training,4.4
9,Purpose & meaning,4.3


In [53]:
# Check responses without comments

df[df["has_comment"] == False][
    ["year", "recommendation_rating", "sentiment", "topics_positive", "topics_negative"]
]

,year,recommendation_rating,sentiment,topics_positive,topics_negative
17,2023,5,NaN,,
21,2023,8,NaN,,
22,2023,9,NaN,,
35,2023,6,NaN,,
37,2023,9,NaN,,
...,...,...,...,...,...
4343,2025,7,NaN,,
4348,2025,8,NaN,,
4362,2025,7,NaN,,
4375,2025,8,NaN,,


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4391 entries, 0 to 4390
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   survey_response_id     4391 non-null   int64 
 1   employee_scd_id        4391 non-null   int64 
 2   year                   4391 non-null   int64 
 3   recommendation_rating  4391 non-null   int64 
 4   comment                3747 non-null   object
 5   has_comment            4391 non-null   bool  
 6   enps_category          4391 non-null   object
 7   topics_positive        4391 non-null   object
 8   topics_negative        4391 non-null   object
 9   sentiment              3747 non-null   object
dtypes: bool(1), int64(4), object(5)
memory usage: 313.2+ KB




---

# PART 6: Creation of the final DataFrame


---



In [56]:
df_final = df.copy()

# Quick check
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4391 entries, 0 to 4390
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   survey_response_id     4391 non-null   int64 
 1   employee_scd_id        4391 non-null   int64 
 2   year                   4391 non-null   int64 
 3   recommendation_rating  4391 non-null   int64 
 4   comment                3747 non-null   object
 5   has_comment            4391 non-null   bool  
 6   enps_category          4391 non-null   object
 7   topics_positive        4391 non-null   object
 8   topics_negative        4391 non-null   object
 9   sentiment              3747 non-null   object
dtypes: bool(1), int64(4), object(5)
memory usage: 313.2+ KB


In [57]:
df_final.head()

,survey_response_id,employee_scd_id,year,recommendation_rating,comment,has_comment,enps_category,topics_positive,topics_negative,sentiment
0,1,202301141,2023,9,Team spirit is strong and the atmosphere is fr...,True,promoter,Pride of belonging; Team climate,,positive
1,2,202300500,2023,7,Pay progression feels slow compared with respo...,True,passive,,Career progression; Compensation,negative
2,3,202301553,2023,8,Innovation and a forward-looking strategy make...,True,passive,Innovation & competitiveness; Management pract...,,positive
3,4,202301348,2023,8,Supportive leadership and clear feedback make ...,True,passive,Management practices,,positive
4,5,202301554,2023,9,Innovation and a forward-looking strategy make...,True,promoter,Innovation & competitiveness; Management pract...,,positive


You can find the output of this dataframe in data > nlp_transformed > 02_sentiment_and_topic_analysis_done.